# Phase 4 - Demo (Colab runner)

Runner only: mount Drive, pull the Phase 4 branch, install gradio, regenerate the summary, then
launch the artifact-backed Gradio demo. Logic lives in `scripts/run_demo.py` and `src/`, not in
this notebook (P1/P2).

The demo degrades gracefully: it launches with **no** `OPENROUTER_API_KEY` (metrics, artifact
views, and BM25 retrieval all work; the answer-generation tab shows disabled). Dense/RRF retrieval
light up only if the embedding stack is installed. Set the key in the optional cell below to enable
grounded answer generation.

## Boot

In [ ]:
# 1. Mount Drive so config.OUTPUT_ROOT (the staged artifacts + chunks) is available.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Get the code onto the VM and pin the Phase 4 branch.
# Repin BRANCH to 'main' once the Phase 4 PRs are merged.
import os

REPO = '/content/FinDocStructRAG'
BRANCH = 'main'  # Phase 4 merged; tracks main

if not os.path.isdir(f'{REPO}/.git'):
    !git clone --quiet https://github.com/AD2000X/FinDocStructRAG.git {REPO}

!cd {REPO} && git fetch origin --quiet
!cd {REPO} && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}
!cd {REPO} && echo branch: $(git rev-parse --abbrev-ref HEAD) HEAD: $(git log --oneline -1)
%cd /content/FinDocStructRAG

## Install demo deps

gradio is a demo-only dependency (not in `requirements-core.txt`).

In [ ]:
!python -m pip install -q gradio

## (Optional) enable answer generation

Leave this cell as-is to run retrieval-only (no key needed). To enable grounded answer generation,
store `OPENROUTER_API_KEY` in Colab Secrets and uncomment the two lines.

In [ ]:
import os
# from google.colab import userdata
# os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
print('answer generation:', 'enabled' if os.getenv('OPENROUTER_API_KEY') else 'disabled (retrieval-only)')

## Build the summary

So the Overview tab and the embedded metrics table are fresh.

In [ ]:
!python scripts/build_phase4_summary.py

## Launch the demo

This cell stays running and prints a public `share` URL. Open it to use the tabs: Overview,
Table QA, Table Extraction, Layout, FUNSD Relations, Limitations. Stop the cell to shut the app
down.

In [ ]:
!python scripts/run_demo.py